# Instruct-Validate-Repair

Use pre- and post-conditions to validate your LLM outputs meet specific requirements.

How can we ensure that the output from a language model is actually good?

Good generative programmers don’t leave this up to chance - instead, they use pre-conditions to ensure that inputs to the LLM are as expected and then check post-conditions to ensure that the LLM’s outputs are fit-for-purpose.

In this recipe, we will use the [Mellea](https://mellea.ai) library to demonstrate using Instruct-Validate-Repair.

Mellea supports several different LLM runtimes, called `backends`, to access an LLM. In this recipe, we will use the [LiteLLM](https://docs.litellm.ai/docs/) [backend](https://github.com/generative-computing/mellea/blob/main/mellea/backends/litellm.py). We can use the LiteLLM backend with providers supported by LiteLLM such as IBM [watsonx.ai](https://www.ibm.com/products/watsonx-ai). In this recipe, you will use the IBM® [Granite](https://www.ibm.com/granite) model now available on watsonx.ai.

# Steps

## Step 1. Set up your environment

While you can choose from several tools, this recipe is best suited for a Jupyter Notebook. Jupyter Notebooks are widely used within data science to combine code with various data sources such as text, images and data visualizations. 

You can run this notebook in [Colab](https://colab.research.google.com/), or download it to your system and [run the notebook locally](https://github.com/ibm-granite-community/granite-kitchen/blob/main/recipes/Getting_Started_with_Jupyter_Locally/Getting_Started_with_Jupyter_Locally.md). 

To avoid Python package dependency conflicts, we recommend setting up a [virtual environment](https://docs.python.org/3/library/venv.html).

Note, this notebook is compatible with Python 3.12, the default in Colab at the time of publishing this recipe. To check your python version, you can run the `!python --version` command in a code cell.


## Step 2. Set up a watsonx.ai instance

See [Getting Started with IBM watsonx](https://github.com/ibm-granite-community/granite-kitchen/blob/main/recipes/Getting_Started/Getting_Started_with_WatsonX.ipynb) for information on getting ready to use watsonx.ai. 

You will need three credentials from the watsonx.ai set up to add to your environment: `WATSONX_URL`, `WATSONX_APIKEY`, and `WATSONX_PROJECT_ID`.

## Step 3. Install relevant libraries and set up credentials and the Mellea session

We'll need a few libraries for this recipe. We will be using Mellea and LiteLLM to use Granite on watsonx.ai.

In [ ]:
! echo "::group::Install Dependencies"
%pip install uv
! uv pip install "git+https://github.com/ibm-granite-community/utils.git" \
    "mellea[litellm]"
! echo "::endgroup::"

Now we will get the credentials to use watsonx.ai and load them into the environment where LiteLLM will expect them.

In [ ]:
from ibm_granite_community.notebook_utils import get_env_var

# Load credentials into environment variables
get_env_var("WATSONX_APIKEY")
get_env_var("WATSONX_PROJECT_ID")
get_env_var("WATSONX_URL")


Now we can create a MelleaSession using the LiteLLM backend connected to the Granite model on watsonx.ai.

In [ ]:
from mellea import start_session, MelleaSession

m: MelleaSession = start_session(
        backend_name="litellm",
        model_id="watsonx/ibm/granite-4-h-small",
    )


## Step 4. Specifying post-conditions using requirements

Suppose that in this case we want to ensure that an email has a salutation and contains only lower-case letters. We can capture these post-conditions by specifying requirements on the Mellea instruct call.

In [ ]:
def write_email_with_requirements(
    m: MelleaSession, name: str, notes: str
) -> str:
    email = m.instruct(
        "Write an email to {{name}} using the notes following: {{notes}}.",
        requirements=[
            "The email should have a salutation",
            "The email must use only lower-case letters",
        ],
        user_variables={"name": name, "notes": notes},
    )
    return str(email)

email = write_email_with_requirements(
        m,
        name="Olivia",
        notes="Olivia helped the lab over the last few weeks by organizing intern events, advertising the speaker series, and handling issues with snack delivery.",
    )

print(email)

We just added two requirements to the instruction which will be added to the model request. By default, the output of the request is checked against the requirements. If all requirements are not satisfied, a second request is sent. In the following examples we show you how you can customize:

* How to define a budget for retrying
* How validation is done for each requirement/check.
* How to set a strategy and (repair) method that is used for retrying

Let’s add a strategy for validating the requirements.

In [ ]:
from mellea.stdlib.sampling import RejectionSamplingStrategy

def write_email_with_strategy(m: MelleaSession, name: str, notes: str) -> str:
    email_candidate = m.instruct(
        "Write an email to {{name}} using the notes following: {{notes}}.",
        requirements=[
            "The email should have a salutation",
            "The email must use only lower-case letters",
        ],
        strategy=RejectionSamplingStrategy(loop_budget=5),
        user_variables={"name": name, "notes": notes},
        return_sampling_results=True,
    )
    for i, loop_results in enumerate(email_candidate.sample_validations, start=1):
        print(f"Loop {i}")
        for j, result in enumerate(loop_results, start=1):
            print(f"Requirement      {j}: {result[0].description}")
            print(f"ValidationResult {j}: {result[1].reason}")

    if email_candidate.success:
        return str(email_candidate.result)
    else:
        print("Expect sub-par result.")
        return str(email_candidate.sample_generations[0])

email = write_email_with_strategy(
        m,
        "Olivia",
        "Olivia helped the lab over the last few weeks by organizing intern events, advertising the speaker series, and handling issues with snack delivery.",
    )

print(email)

A couple of things happened here. First, we added a sampling strategy to the instruction. This strategy (`RejectionSamplingStrategy`) checks if all requirements are met. If any requirement fails, then the sampling strategy will sample a new email from the LLM. This process will repeat until the `loop_budget` on retries is consumed or all requirements are met.

Even with retries, sampling might not generate results that fulfill all requirements (`email_candidate.success==False`). Mellea forces you to think about what it means for an LLM call to fail; in this case, we handle the situation by simply returning the first sample as the final result.

<div class="alert alert-block alert-info">
ℹ️ 
When using the <code>return_sampling_results=True</code> parameter, the <code>instruct</code> function returns a <code>SamplingResult</code> object (not a <code>ModelOutputThunk</code>) which carries the full history of sampling and validation results for each sample.
</div>

## Step 5. Validating Requirements

Now that we defined requirements and sampling we should have a look into **how requirements are validated**. The default validation strategy is **LLM-as-a-judge**.

Let’s look on how we can customize requirement definitions.

In [ ]:
from mellea.stdlib.requirement import check, req, simple_validate

requirements = [
            req("The email should have a salutation"), # r1
            req(
                "The email must use only lower-case letters",
                validation_fn=simple_validate(lambda x: x.lower() == x),
            ), # r2
            check("Do not mention purple elephants.") # r3
        ]


Here, the first requirement (r1) will be validated by LLM-as-a-judge on the output (last turn) of the instruction. This is the default behavior, since nothing else is specified.

The second requirement (r2) simply uses a function that takes the output of a sampling step and returns a boolean value indicating validation was successful or not. While the `validation_fn` parameter expects to run validation on the full session context (see [Context Management](https://docs.mellea.ai/core-concept/context-management)), Mellea provides a wrapper for simpler validation functions (`simple_validate(fn: Callable[[str], bool])`) that take the output string and return a boolean as seen in this case.

The third requirement is a `check`. Checks are only used for validation, not for generation. Checks aim to avoid the “do not think about B” effect that often primes models (and humans) to do the opposite and “think” about B.

<div class="alert alert-block alert-info">
ℹ️ 
LLM-as-a-Judge is not presumptively robust. Whenever possible, implement requirement validation using plain old Python code. When a model is necessary, it can often be a good idea to train a <code>calibrated</code> model specifically for your validation problem. <a href="https://docs.mellea.ai/core-concept/adapters">Adapters</a> explains how to use Mellea’s <code>m tune</code> subcommand to train your own LoRAs for requirement checking (and for other types of Mellea components as well).
</div>

In [ ]:

def write_email_with_strategy(m: MelleaSession, name: str, notes: str) -> str:
    email_candidate = m.instruct(
        "Write an email to {{name}} using the notes following: {{notes}}.",
        requirements=requirements, # type: ignore
        strategy=RejectionSamplingStrategy(loop_budget=5),
        user_variables={"name": name, "notes": notes},
        return_sampling_results=True,
    )
    for i, loop_results in enumerate(email_candidate.sample_validations, start=1):
        print(f"Loop {i}")
        for j, result in enumerate(loop_results, start=1):
            print(f"Requirement      {j}: {result[0].description}")
            print(f"ValidationResult {j}: {result[1].reason}")

    if email_candidate.success:
        return str(email_candidate.result)
    else:
        print("Expect sub-par result.")
        return str(email_candidate.sample_generations[0])

email = write_email_with_strategy(
        m,
        "Olivia",
        "Olivia helped the lab over the last few weeks by organizing intern events, advertising the speaker series, and handling issues with snack delivery.",
    )

print(email)

## Step 6. Conclusion

We have used the generative programming design pattern: **Instruct - Validate - Repair (IVR)**.

When LLMs work well, the software developer experiences the LLM as a sort of oracle that can handle most any input and produce a sufficiently desirable output. When LLMs do not work at all, the software developer experiences the LLM as a naive markov chain that produces junk. In both cases, the LLM is just sampling from a distribution.

The crux of generative programming is that most applications find themselves somewhere in-between these two extremes — the LLM mostly works, enough to demo a tantilizing MVP. But failure modes are common enough and severe enough that complete automation is beyond the developer’s grasp.

Traditional software deals with failure modes by carefully describing what can go wrong and then providing precise error handling logic. When working with LLMs, however, this approach suffers a Sysiphean curse. There is always one more failure mode, one more special case, one more new feature request.